<a href="https://colab.research.google.com/github/chisamitsu/ita-dataanalytics/blob/main/s07-python-basico/s07-python-basico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# S07 – Python básico: funciones, estructuras de datos y bucles

Evidencias de ejecución – IT Academy, Data Analytics

| | |
|---|---|
| **Nombre** | Carlos Hisamitsu |
| **Fecha** | [DD/MM/AAAA] |
| **Repositorio GitHub** | https://github.com/chisamitsu/ita-dataanalytics/tree/main/s07-python-basico |

### Librerías

In [51]:
import re

# NIVEL 1

## Ejercicio 1 – Selección de personal

### Enunciado

> Tu treballes com a analista de dades en el departament de RRHH d'una empresa. S'ha obert una nova oferta laboral interna i l'empresa ha decidit que només les primeres 10 persones que s'inscriguin podran participar en el procés de selecció.

Per això, t'han encarregat desenvolupar una eina senzilla en Python que actuï com a formulari d'inscripció i sistema de classificació inicial. Aquest formulari ha de permetre introduir, per a cada candidat/a:

- Nom i Cognoms
- Telèfon de contacte
- Anys d'experiència dins de l'empresa
- Coneixement de Català (0–3 punts)
  - 0 = sense coneixement
  - 1 = fins a nivell B1
  - 2 = nivell B2
  - 3 = C1 o superior
- Domini d'Excel (0–3 punts)
  - 0 = nivell bàsic
  - 1 = ús de funcions
  - 2 = taules dinàmiques
  - 3 = Visual Basic i Macros
- Títol acadèmic (0–3 punts)
  - 0 = ESO
  - 1 = FP
  - 2 = Universitat
  - 3 = Màster o superior

Assegurant-se que tota la informació introduïda és correcta i entra dins els valors permesos. A partir d'aquestes dades, el programa haurà de calcular una puntuació total aplicant el pes de l'experiència (anys * 1.5) i sumant-hi els punts de català, excel i titulació, i a continuació classificar automàticament cada persona segons els llindars establerts per l'empresa:

>- Prioritat Alta → puntuació ≥ 12
>- Candidat adequat → 8 ≤ puntuació < 12
>- No prioritari → puntuació < 8

Cada vegada que algú complet a el formulari, s'ha de mostrar la seva puntuació individual. Quan ja s'hagin inscrit exactament deu persones, l'eina ha de deixar d'acceptar nous registres i generar un informe final en pantalla, ordenat primer segons la prioritat i, en cas d'empat, per la puntuació obtinguda. Aquest informe ha d'incloure el nom, el telèfon i la puntuació de cada candidat per tal que el departament de RRHH els pugui contactar ràpidament.

#### **Important**

Per resoldre aquest exercici, hauràs de:

- Desenvolupar un sistema modular amb les funcions següents:
  - obtenir_dades
  - validar_dades
  - calcular_puntuacio
  - classificar_candidat
  - mostrar_individual
  - mostrar_informe
  - main_seleccio # orquestra totes les funcions
- Escriure primer l'algoritme pas a pas, indicant clarament quines operacions farà cada funció.
- Escriure el codi de cada funció.

Per fer proves:

```
inputs_test = [
    "Laura Serra", "608112233", 6, 3, 3, 3,
    "Clara Gómez", "655221199", 3, 3, 3, 2,
    "Sergio López", "644112211", 3, 2, 2, 2,
    "Candidat Error", "611000000", 5, 7, 3, 2, # Català = 7 → NO VÀLID
    "Marta Ruiz", "699887766", 2, 3, 2, 1,
    "Marc Vidal", "612443322", 6, 3, 3, 3,
    "Júlia Romero", "677998877", 5, 3, 3, 2,
    "Pol Navarro", "699554433", 4, 3, 2, 2,
    "David Ferrer", "677665544", 2, 1, 1, 1,
    "Anna Paredes", "633221144", 1, 2, 1, 0,  
    "Anibal Paredes", "644272727", 1, 2, 1, 0,  
    "Núria Casals", "600112200", 8, 2, 3, 3 # 11è candidat     
]
```

### Algoritmo

#### Proceso (según el enunciado)
- Formulario:
  - obtener los datos
  - comprobar/validar los datos
- Calcular los puntos
- Definir los grupos de prioridad
- Mostrar los puntos calculados de cada candidato
- Máximo 10 inscripciones, después se cierra el formulario
- Mostrar el informe
  - ordenado por prioridad y después por puntos

#### Funciones

- **obtenir_dades** (`get_data`): recoge los datos de cada campo del formulario. Para asegurar que los datos son correctos, valida cada respuesta justo después de introducirla y vuelve a pedirla si no es válida. El usuario puede escribir "cancel" para descartar el candidato. Los datos del candidato se guardan en un diccionario.
- **validar_dades** (`validate_data`): función reutilizable que valida cada respuesta según los requisitos de su campo. Devuelve el valor (limpiado o convertido si hace falta, por ejemplo quitando espacios o convirtiendo números) o `None` si el dato es incorrecto, y también un mensaje de error, si lo hay.
- **calcular_puntuacio** (`candidate_score`): calcula los puntos con la fórmula definida.
- **classificar_candidat** (`candidate_tier`): asigna la categoría del candidato según su puntuación.
- **mostrar_individual** (`print_individual`): busca al candidato en la lista y muestra su puntuación y su categoría.
- **mostrar_informe** (`print_report`): muestra en pantalla el nombre, el teléfono y los puntos de los candidatos, ordenados por prioridad y puntos.
- **main_seleccio** (`run_registration`): orquesta las funciones para coordinar el proceso completo.

### Implementación

#### Preparación
Para facilitar el mantenimiento y poder definir las funciones de forma más dinámica, vamos a crear una lista con las preguntas, el tipo de respuesta esperada y las reglas de validación.

También definimos una (pseudo) constante con la cantidad máxima de candidatos y una lista vacía de candidatos como variable global (he considerado un diccionario de candidatos, pero podría haber candidatos con el mismo nombre, por lo que decidí usar una lista).

In [52]:
MAX_CANDIDATES = 10

PHONE_REGEX = r"^\+?\d{7,15}$"

# the list of candidates is a global variable
candidates = []

# define the list of questions, input type and data validation
form_questions = [
    {
        "id": "full_name",
        "input_label": "Nom i Cognoms",
        "type": "text",
        "len_min": 2,
        "len_max": 50,
        # at least one letter; letters, spaces, dots, apostrophes, hyphens and middle dot (l·l)
        "regex": r"^(?=.*[A-Za-zÀ-ÖØ-öø-ÿ])[A-Za-zÀ-ÖØ-öø-ÿ·.' -]{2,50}$",
        "regex_message": "El valor només pot contenir lletres, espais, punts, apòstrofs, guions i punt volat"
    },{
        "id": "phone",
        "input_label": "Telèfon de contacte",
        "type": "phone"
    },{
        "id": "seniority",
        "input_label": "Anys d'experiència dins de l'empresa",
        "type": "float",
        "range_min": 0,
        "range_max": 50
    },{
        "id": "level_catalan",
        "input_label": "Coneixement de Català",
        "type": "int",
        "range_min": 0,
        "range_max": 3,
        "tip": "0=Sense coneixement | 1=B1 | 2=B2 | 3=C1 o superior"
    },{
        "id": "level_excel",
        "input_label": "Domini d'Excel",
        "type": "int",
        "range_min": 0,
        "range_max": 3,
        "tip": "0=Bàsic | 1=Funcions | 2=Taules dinàmiques | 3=VBA i Macros"
    },{
        "id": "level_education",
        "input_label": "Títol acadèmic",
        "type": "int",
        "range_min": 0,
        "range_max": 3,
        "tip": "0=ESO | 1=FP | 2=Universitat | 3=Màster o superior"
    }
]

#### Funciones

##### **validate_data** (validar_dades)
Recibe como parámetro el diccionario "question" con los atributos esperados para el tipo de dato. Además, recibe el valor de respuesta del usuario para validar.

1. Uso el método ```question.get()``` para controlar posibles valores no existentes.
1. De acuerdo con el tipo de dato esperado (text, int, float, phone) la función aplica las validaciones definidas.
1. Si es posible, los datos se tratan para poder usarlos (elimina espacios en blanco al inicio o final del texto, convierte números, acepta la coma como separador decimal, etc.).
1. Si se encuentra algún error en la validación, se define un mensaje de error.
1. La función retorna el valor de respuesta formateado (si es posible) y el mensaje de error (si existe). Siempre habrá solamente un valor de retorno y el otro será ```None```.

In [53]:
# Validates a data input based on the type and requirements defined in the questions dictionary
def validate_data(question, answer):
  error_type = None
  error_message = None

  # get requirements, if they exist
  question_type = question.get("type")
  len_min = question.get("len_min", 1)   # text answers must not be empty by default
  len_max = question.get("len_max")
  range_min = question.get("range_min")
  range_max = question.get("range_max")
  regex = question.get("regex")
  regex_message = question.get("regex_message","El valor no és vàlid")

  # validate text type
  if question_type == "text":
    answer = answer.strip()
    # validate length, each bound is optional (len_min is 1 if not defined)
    if (len(answer) < len_min or (len_max is not None and len(answer) > len_max)):
      error_type = "length"
    # in case a regular expression was defined, then validate it
    elif (regex is not None and not re.match(regex, answer)):
      error_type = "regex"

  # validate integer type
  elif question_type == "int":
    try:
      answer = int(answer)
      # only validate range if both range_min and range_max are defined
      if (range_min is not None and range_max is not None and not range_min <= answer <= range_max):
        error_type = "range"
    except ValueError:
      error_type = "type int"

  # validate float type (a comma is accepted as decimal separator)
  elif question_type == "float":
    try:
      answer = float(answer.replace(",", "."))
      # only validate range if both range_min and range_max are defined
      if (range_min is not None and range_max is not None and not range_min <= answer <= range_max):
        error_type = "range"
    except ValueError:
      error_type = "type float"

  # validate phone type
  elif question_type == "phone":
    # clean up unnecessary characters that are common on phone info
    answer = answer.strip()
    answer = (answer.replace(" ", "").replace("-", ""))

    if not re.match(PHONE_REGEX, answer):
      error_type = "phone"

  # if any error was found, then clean the answer value as it is not valid
  if error_type is not None:
    answer = None

  # set the error message, if any error was detected
  if error_type == "length":
    if len_max is None:
      error_message = f"El valor ha de tenir com a mínim {len_min} caràcters"
    else:
      error_message = f"El valor ha de tenir entre {len_min} i {len_max} caràcters"
  elif error_type == "range":
    error_message = f"El valor ha de ser entre {range_min} i {range_max}"
  elif error_type == "type int":
    error_message = "El valor ha de ser un número enter"
  elif error_type == "type float":
    error_message = "El valor ha de ser un número (es poden usar decimals)"
  elif error_type == "phone":
    error_message = "El valor ha de ser un número de telèfon vàlid"
  elif error_type == "regex":
    error_message = regex_message

  return answer, error_message

##### **get_data** (obtenir_dades)
Para cada pregunta en la lista ```form_questions```:
1. Escribe la pregunta.
1. Si hay pista para la pregunta, la escribe debajo.
1. El usuario responde la pregunta.
1. Si la respuesta es "cancel", descarta el candidato y retorna ```None```.
1. Valida la respuesta de acuerdo con el tipo de dato (```validate_data()```).
1. Si hay algún error, muestra el mensaje y espera un nuevo valor. Repite hasta que el dato sea correcto.
1. Si no hay error, guarda el dato en el diccionario del candidato (```candidate```).

In [54]:
# Gets the candidate's data and returns it as a dictionary (None if the registration is cancelled)
def get_data():
  candidate = {}
  for question in form_questions:
    print(f"\n{question['input_label']}")
    tip = question.get("tip")
    if tip:
        print(f"{tip}")

    while True:
      answer = input("> ")

      if answer.strip().lower() == "cancel":
        return None

      answer, error_message = validate_data(question, answer)
      if error_message:
        print(error_message)
        print("Introdueix un valor vàlid o escriu 'cancel' per descartar el candidat")
      else:
        candidate[question['id']] = answer
        break

  return candidate

##### **candidate_score** (calcular_puntuacio)
Una función sencilla para calcular la puntuación del candidato.
1. Recibe como parámetro los datos del candidato.
1. Busca los valores necesarios para calcular.
1. Calcula con la fórmula ```(seniority * 1.5) + level_catalan + level_excel + level_education```.
1. Retorna el resultado.

In [55]:
def candidate_score(candidate):
  seniority = candidate.get("seniority",0)
  level_catalan = candidate.get("level_catalan",0)
  level_excel = candidate.get("level_excel",0)
  level_education = candidate.get("level_education",0)

  score = (seniority * 1.5) + level_catalan + level_excel + level_education
  return score

##### **candidate_tier** (classificar_candidat)
Una función sencilla que recibe como parámetro los puntos del candidato y retorna la clasificación.
1. Si puntos ≥ 12: Prioritat Alta
1. Si 8 ≤ puntos < 12: Candidat adequat
1. Si puntos < 8: No prioritari

In [56]:
def candidate_tier(score):
  tier = ""
  if score >= 12:
    tier = "Prioritat Alta"
  elif score >= 8:
    tier = "Candidat adequat"
  else:
    tier = "No prioritari"
  return tier

##### **print_individual** (mostrar_individual)
Recibe como parámetro el nombre del candidato y muestra en pantalla sus puntos, si lo encuentra.

La búsqueda se hace desde el final de la lista: así, si hay nombres repetidos, se muestra el candidato que se acaba de registrar.

In [57]:
def print_individual(candidate_name):
  # search from the end: the candidate just registered is the last one
  for candidate in reversed(candidates):
    if candidate["full_name"] == candidate_name:
      print(f"\nCandidat: {candidate_name}")
      print(f"Puntuació: {candidate['score']}")
      print(f"Prioritat: {candidate['tier']}")
      break
  return

##### **print_report** (mostrar_informe)
Muestra en pantalla los datos de los candidatos registrados, ordenados por prioridad y puntos. El ejercicio pide ordenar por prioridad y después por puntos; como la prioridad se calcula a partir de los puntos, ordenar por puntos es suficiente.

Muestra el nombre, el teléfono y los puntos. La columna numérica se alinea a la derecha.

In [58]:
def print_report():
  candidates.sort(key=lambda x: (x["score"]), reverse=True)

  # column widths: the minimum needed for the header and the longest value
  headers = ("Nom i Cognoms", "Telèfon", "Punts")
  alignments = ["<", "<", ">"]
  rows = [(c["full_name"], c["phone"], f"{c['score']:.1f}") for c in candidates]
  widths = [max(len(value) for value in column) for column in zip(headers, *rows)]
  line = "-" * (sum(widths) + 3 * (len(widths) - 1))

  print("\nInforme de candidats\n")
  print(line)
  print(" | ".join(f"{header:<{width}}" for header, width in zip(headers, widths)))
  print(line)

  for row in rows:
    print(" | ".join(f"{value:{align}{width}}" for value, width, align in zip(row, widths, alignments)))

  return

##### **run_registration** (main_seleccio)

Coordina la ejecución de la inscripción de candidatos.

- El número total de candidatos está definido en la variable global ```MAX_CANDIDATES```.
- Empieza mostrando el número del candidato que será registrado.
- Llama a la función para recopilar los datos del candidato ```get_data()```.
- Si los datos del candidato son correctos, calcula la puntuación y la prioridad, y guarda los datos en la lista de candidatos. Si el registro se cancela, el candidato se descarta y no cuenta para el máximo.
- Cuando llega al número máximo de candidatos, cierra las inscripciones y muestra el informe ordenado.

In [59]:
def run_registration():
  # create the candidates list
  global candidates
  candidates = []

  while len(candidates) < MAX_CANDIDATES:
    print("=" * 50)
    print(f"\nInscriu el candidat {len(candidates) + 1}")
    candidate = get_data()
    if candidate:
      candidate["score"] = candidate_score(candidate)
      candidate["tier"] = candidate_tier(candidate["score"])
      candidates.append(candidate)

      print_individual(candidate["full_name"])
      print()
    else:
      print("\nCandidat descartat\n")

  print("=" * 50)
  print("\nInscripcions tancades\n")
  print("=" * 50)

  print_report()
  print()
  print("=" * 50)
  return

### Ejecución

Para la prueba uso los datos de `inputs_test` a través de una función `input()` que va leyendo la lista, así se ve en pantalla lo que se "escribe" en cada pregunta. Cambié la fila de "Candidat Error": después del 7 (que da error de rango) añadí "cancel" para que se vea cómo se descarta el candidato y se pasa al siguiente. Con los datos originales, el formulario volvería a pedir el valor y se desordenaría el resto.

Se registran 10 candidatos (el descartado no cuenta), y "Núria Casals", que sería el 11º, ya no se pregunta porque las inscripciones se cierran. El informe sale ordenado por puntos.

In [60]:
inputs_test = [
    "Laura Serra", "608112233", 6, 3, 3, 3,
    "Clara Gómez", "655221199", 3, 3, 3, 2,
    "Sergio López", "644112211", 3, 2, 2, 2,
    "Candidat Error", "611000000", 5, 7, "cancel",   # 7 → error, then cancel
    "Marta Ruiz", "699887766", 2, 3, 2, 1,
    "Marc Vidal", "612443322", 6, 3, 3, 3,
    "Júlia Romero", "677998877", 5, 3, 3, 2,
    "Pol Navarro", "699554433", 4, 3, 2, 2,
    "David Ferrer", "677665544", 2, 1, 1, 1,
    "Anna Paredes", "633221144", 1, 2, 1, 0,
    "Anibal Paredes", "644272727", 1, 2, 1, 0,
    "Núria Casals", "600112200", 8, 2, 3, 3,         # 11th: never asked
]

test_values = iter(inputs_test)

# replaces the built-in input() only for this test run
def input(prompt=""):
    value = str(next(test_values))
    print(f"{prompt}{value}")
    return value

run_registration()

del input   # restores the built-in input()


Inscriu el candidat 1

Nom i Cognoms
> Laura Serra

Telèfon de contacte
> 608112233

Anys d'experiència dins de l'empresa
> 6

Coneixement de Català
0=Sense coneixement | 1=B1 | 2=B2 | 3=C1 o superior
> 3

Domini d'Excel
0=Bàsic | 1=Funcions | 2=Taules dinàmiques | 3=VBA i Macros
> 3

Títol acadèmic
0=ESO | 1=FP | 2=Universitat | 3=Màster o superior
> 3

Candidat: Laura Serra
Puntuació: 18.0
Prioritat: Prioritat Alta


Inscriu el candidat 2

Nom i Cognoms
> Clara Gómez

Telèfon de contacte
> 655221199

Anys d'experiència dins de l'empresa
> 3

Coneixement de Català
0=Sense coneixement | 1=B1 | 2=B2 | 3=C1 o superior
> 3

Domini d'Excel
0=Bàsic | 1=Funcions | 2=Taules dinàmiques | 3=VBA i Macros
> 3

Títol acadèmic
0=ESO | 1=FP | 2=Universitat | 3=Màster o superior
> 2

Candidat: Clara Gómez
Puntuació: 12.5
Prioritat: Prioritat Alta


Inscriu el candidat 3

Nom i Cognoms
> Sergio López

Telèfon de contacte
> 644112211

Anys d'experiència dins de l'empresa
> 3

Coneixement de Català
0=S

### Interpretación de los resultados

Se registraron los 10 candidatos. El de "Candidat Error" se descartó con "cancel" después del error con el valor "7", y "Núria Casals" no llegó a preguntarse porque ya estaban cerradas las inscripciones.

Cinco quedaron en Prioritat Alta, dos en Candidat adequat y tres en No prioritari. La experiencia es lo que más pesa en la puntuación, por eso Laura y Marc (6 años) están arriba con 18.0 (pero esos candidatos también son los que tienen más años de experiencia).

Hay dos empates (Laura y Marc, Anna y Anibal) y el informe los deja en el orden de registro.